In [ ]:
import sys
import os
import numpy as np
import scipy.linalg as la
from scipy.signal import cont2discrete
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D 
import matplotlib as mpl

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.config as cfg
from utils.quadcopter_model import quad_hover_linear_model, create_discrete_model, gaussian_dist, gaussian_dist_time_varying, gaussian_wind_direction, visualize_gaussian_samples, visualize_multi_task_gaussians, visualize_multi_task_gaussians_3D, plot_disturbances_time, animate_trajectories, plot_angles, plot_xyz, plot_controls_grid, plot_angles_mean, plot_xyz_mean, plot_controls_grid_mean, summarize_checks
from utils.simple_MLP import PolicyMLP, rollout_policy, traj_cost, train_nn_policy, meta_train_nn_policy, maml_adapt, evaluate_policy_cost, plot_maml_diagnostics, plot_adapted_losses, plot_adaptation_comparison, plot_loss_evolution

# Plotting parameters
plt.style.use('default')
plt.rcParams['figure.dpi'] = 400

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"], 
    "axes.unicode_minus": False 
})

# Consistent printing of numpy arrays
np.set_printoptions(precision=3, suppress=True)

In [2]:
# Create and discretize model
A, B, C, D = quad_hover_linear_model()
# print("A =\n", A, "\n\nB =\n", B, "\n\nC =\n", C, "\n\nD =\n", D)

A_d, B_tilde_d, C_d, D_d = create_discrete_model(A, B, C, D)
B_d = B_tilde_d[:, :4]
# print("A_d =\n", A_d, "\n\nB_tilde_d =\n", B_tilde_d, "\n\nC_d =\n", C_d, "\n\nD_d =\n", D_d)

In [12]:
np.random.seed(5)

# TRAINING TASKS: directions only in allowed quadrants
point_per_task = 50
num_tasks = 6

tasks_all_disturbances = []
tasks_all_x0 = []
tasks_dx = []
tasks_dy = []

training_angles = []  # to store angles for the vector plot
training_means  = []  # amplitudes for vector plot

def sample_training_angle():
    """Sample angle only in quadrants I, II, and IV."""
    while True:
        ang = np.random.uniform(0, 360)
        if (0 <= ang < 180) or (270 <= ang <= 360):
            return ang

print("=== TRAINING TASKS ===")
for i in range(num_tasks):

    # amplitude parameters
    mean  = np.random.uniform(0, 6.0)
    sigma = np.random.uniform(0.1, 0.6)

    # allowed wind direction quadrants
    direction_deg = sample_training_angle()

    # generate disturbances
    dx, dy = gaussian_wind_direction(
        mean=mean,
        sigma=sigma,
        N=point_per_task,
        direction_deg=direction_deg,
        device="cpu"
    )

    dz = -cfg.G * torch.ones_like(dx)
    disturbances = torch.cat([dx, dy, dz], dim=-1)

    tasks_all_disturbances.append(disturbances)
    tasks_dx.append(dx)
    tasks_dy.append(dy)

    x0 = torch.zeros(point_per_task, 12)
    tasks_all_x0.append(x0)

    training_angles.append(direction_deg)
    training_means.append(mean)

    print(f"[Task {i}] mean={mean:.2f}, sigma={sigma:.2f}, direction={direction_deg:.1f}°")

# Visualization of xyz distributions
visualize_multi_task_gaussians_3D(tasks_all_disturbances, filename="results_all_directions/training_dist.png")

# Joint training dataset
tasks_all_disturbances_joint = torch.cat(tasks_all_disturbances, dim=0)
tasks_all_x0_joint = torch.cat(tasks_all_x0, dim=0).float()


# TESTING TASK: direction ONLY in quadrant III (180° to 270°)
print("\n=== TESTING TASK ===")
point_per_task = 50

wind_mean  = np.random.uniform(0, 6.0)
wind_sigma = np.random.uniform(0.1, 0.6)
wind_angle = np.random.uniform(180, 270)   # QUADRANT III ONLY

dx_test, dy_test = gaussian_wind_direction(
    mean=wind_mean,
    sigma=wind_sigma,
    N=point_per_task,
    direction_deg=wind_angle,
    device="cpu"
)

dz_test = -cfg.G * torch.ones_like(dx_test)
disturbances_test = torch.cat([dx_test, dy_test, dz_test], dim=-1)
x0_test = torch.zeros(point_per_task, 12)

testing_angle = wind_angle
testing_mean  = wind_mean

visualize_multi_task_gaussians_3D([disturbances_test], filename="results_all_directions/testing_dist.png")

print(f"[Testing task] mean={wind_mean:.2f}, sigma={wind_sigma:.2f}, direction={wind_angle:.1f}°")

=== TRAINING TASKS ===
[Task 0] mean=1.33, sigma=0.54, direction=74.4°
[Task 1] mean=5.87, sigma=0.27, direction=134.0°
[Task 2] mean=1.43, sigma=0.46, direction=286.6°
[Task 3] mean=0.71, sigma=0.13, direction=321.8°
[Task 4] mean=2.95, sigma=0.29, direction=273.7°
[Task 5] mean=5.15, sigma=0.36, direction=64.8°

=== TESTING TASK ===
[Testing task] mean=5.92, sigma=0.32, direction=259.9°


In [251]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch
from matplotlib.legend_handler import HandlerPatch

class HandlerArrow(HandlerPatch):
    def __init__(self, arrowstyle='->', mutation_scale=15, lw=2, **kwargs):
        super().__init__()
        self.arrowstyle = arrowstyle
        self.mutation_scale = mutation_scale
        self.lw = lw

    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):

        arrow = FancyArrowPatch(
            (0, height/2), (width, height/2),
            arrowstyle=self.arrowstyle,
            color=orig_handle.get_edgecolor(),
            mutation_scale=self.mutation_scale,
            lw=self.lw,
            joinstyle='miter',
            capstyle='butt',  
        )
        arrow.set_transform(trans)
        return [arrow]

def plot_wind_vectors(training_angles, training_means, testing_angle, testing_mean, filename):

    plt.figure(figsize=(5,5))

    # --- TRAINING VECTORS ---
    for ang, amp in zip(training_angles, training_means):
        x = amp * np.cos(np.deg2rad(ang))
        y = amp * np.sin(np.deg2rad(ang))
        plt.arrow(0, 0, x, y,
                  head_width=0.2, lw=1.5, length_includes_head=True,
                  color="blue", alpha=0.9)

    # --- TESTING VECTOR ---
    xt = testing_mean * np.cos(np.deg2rad(testing_angle))
    yt = testing_mean * np.sin(np.deg2rad(testing_angle))
    plt.arrow(0, 0, xt, yt,
              head_width=0.2, lw=1.5, length_includes_head=True,
              color="red", alpha=0.9)

    # Create dummy arrow handles for legend
    train_arrow = FancyArrowPatch((0,0), (1,0), color="blue")
    test_arrow  = FancyArrowPatch((0,0), (1,0), color="red")

    plt.legend(
        [train_arrow, test_arrow],
        ["Training wind", "Testing wind"],
        handler_map={
            FancyArrowPatch: HandlerArrow(
                arrowstyle='-|>',   # nicer arrow head
                mutation_scale=8,  # bigger arrowhead
                lw=1.5              # thicker line
            )
        },
        loc="upper left"
    )

    # Formatting
    plt.axhline(0, color="black", linewidth=0.8)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel(r"$d_x$")
    plt.ylabel(r"$d_y$")
    plt.title("Wind Direction Vectors (mean amplitude)")
    plt.grid(True, alpha=0.3, linestyle='-.', zorder=0)
    plt.xlim(-6, 6)
    plt.ylim(-6, 6)
    plt.gca().set_aspect("equal", adjustable="box")

    plt.savefig(filename, dpi=300)
    plt.close()

plot_wind_vectors(training_angles, training_means, testing_angle, testing_mean, "results_all_directions/wind_vectors.png")

In [ ]:
# JOINT TRAINING (BASELINE)
device = "cpu"
epochs = 5000
compute_mean_every = 100
lr = 1e-3

joint_policy, train_means = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=tasks_all_x0_joint, disturbances=tasks_all_disturbances_joint,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=True
)

/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:199: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:200: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)


[ep 100/5000]:   train_loss=9.3215e+04   |    eval_loss=7.2278e+04   |   |x_T|_avg=65.557
[ep 200/5000]:   train_loss=2.6983e+04   |    eval_loss=2.6048e+04   |   |x_T|_avg=28.495
[ep 300/5000]:   train_loss=1.6142e+04   |    eval_loss=1.5867e+04   |   |x_T|_avg=25.307
[ep 400/5000]:   train_loss=7.7695e+03   |    eval_loss=7.9766e+03   |   |x_T|_avg=18.147
[ep 500/5000]:   train_loss=5.0950e+03   |    eval_loss=5.0652e+03   |   |x_T|_avg=12.395
[ep 600/5000]:   train_loss=3.5437e+03   |    eval_loss=3.5647e+03   |   |x_T|_avg=8.926
[ep 700/5000]:   train_loss=2.3140e+03   |    eval_loss=2.3062e+03   |   |x_T|_avg=8.048
[ep 800/5000]:   train_loss=9.7830e+02   |    eval_loss=9.6261e+02   |   |x_T|_avg=3.428
[ep 900/5000]:   train_loss=2.5705e+02   |    eval_loss=2.5563e+02   |   |x_T|_avg=0.869
[ep 1000/5000]:   train_loss=1.2053e+02   |    eval_loss=7.0310e+02   |   |x_T|_avg=1.032
[ep 1100/5000]:   train_loss=8.8120e+01   |    eval_loss=8.8189e+01   |   |x_T|_avg=0.352
[ep 1200/5000]

In [323]:
def plot_loss_evolution(train_means, total_epochs, log_interval, filename="training_loss_evolution.png"):
    plt.figure(figsize=(7, 5))

    epochs_range = np.arange(log_interval, total_epochs + 1, log_interval)

    plt.plot(epochs_range, train_means, 'blue', alpha=0.8)
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Mean training loss")
    plt.grid(True, which="both", ls="--", lw=0.4)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()
plot_loss_evolution(train_means, epochs, compute_mean_every, filename="results_all_directions/loss_joint_training.png")

In [14]:
# META TRAINING
device = "cpu"
epochs = 5000
compute_mean_every = 100
lr_outer_loop = 1e-3            # β
lr_inner_loop = 1e-3            # α
nb_samples_inner_training = 5
nb_samples_inner_testing = 5


meta_policy, logs = meta_train_nn_policy(
    A_d_nt = A_d, B_tilde_d_nt = B_tilde_d,
    init_points_list = tasks_all_x0,          
    disturbances_list = tasks_all_disturbances,
    lr_outer=lr_outer_loop,          
    lr_inner=lr_inner_loop,   
    meta_batch_size=None, 
    K_inner=nb_samples_inner_training,               
    K_outer=nb_samples_inner_testing,    
    epochs=epochs,
    device="cpu",
    verbose_every=compute_mean_every
)

[MAML] Meta-training on 6 tasks.


/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:373: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(theta_policy, theta_i_prime, (x,))


[ep 100/5000] meta_loss = 4.2469e+05
[ep 200/5000] meta_loss = 1.4953e+05
[ep 300/5000] meta_loss = 8.7896e+04
[ep 400/5000] meta_loss = 6.4485e+04
[ep 500/5000] meta_loss = 5.8503e+04
[ep 600/5000] meta_loss = 4.1856e+04
[ep 700/5000] meta_loss = 3.7353e+04
[ep 800/5000] meta_loss = 3.6151e+04
[ep 900/5000] meta_loss = 2.4503e+04
[ep 1000/5000] meta_loss = 1.7196e+04
[ep 1100/5000] meta_loss = 9.4930e+03
[ep 1200/5000] meta_loss = 7.6953e+03
[ep 1300/5000] meta_loss = 6.8100e+03
[ep 1400/5000] meta_loss = 5.6860e+03
[ep 1500/5000] meta_loss = 4.8349e+03
[ep 1600/5000] meta_loss = 4.0951e+03
[ep 1700/5000] meta_loss = 3.2782e+03
[ep 1800/5000] meta_loss = 2.6766e+03
[ep 1900/5000] meta_loss = 1.8554e+03
[ep 2000/5000] meta_loss = 8.4860e+02
[ep 2100/5000] meta_loss = 3.3779e+02
[ep 2200/5000] meta_loss = 2.4053e+02
[ep 2300/5000] meta_loss = 1.7995e+02
[ep 2400/5000] meta_loss = 1.4604e+02
[ep 2500/5000] meta_loss = 1.2768e+02
[ep 2600/5000] meta_loss = 1.0972e+02
[ep 2700/5000] meta_l

In [ ]:
plot_maml_diagnostics(logs, filename="results_all_directions/loss_meta_training.png")

In [316]:
def plot_adapted_losses(adapt_losses, full_mean_losses, full_min_losses, full_max_losses, filename):
    steps = np.arange(0, len(full_mean_losses))

    plt.figure(figsize=(6,4))
    plt.plot(steps, adapt_losses, marker='o', markersize=4, color="red", alpha=0.8)
    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Inner Loss (5 samples)")
    plt.title("Inner Adaptation Loss (used for weight updates)")
    plt.grid(True)
    plt.xlim(0, 5)
    plt.tight_layout()
    plt.savefig(filename.replace(".png", "_inner.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(6,4))
    plt.plot(steps, full_mean_losses, marker='o', markersize=4, label="Mean Loss", color="red", alpha=0.8)
    plt.fill_between(
        steps,
        full_min_losses,
        full_max_losses,
        alpha=0.15,
        edgecolor="none",
        color="red",
        label="Min-Max Range"
    )

    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Full Test Loss")
    plt.title("Performance on Entire Test Distribution")
    plt.grid(True)
    plt.xlim(0, 5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename.replace(".png", "_full.png"), dpi=300)
    plt.close()


def plot_adaptation_comparison(full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, filename):
    steps = np.arange(0, len(full_mean_losses_joint))
    plt.figure(figsize=(6,4))

    color_meta = "#D81B60"
    plt.plot(steps, full_mean_losses_meta, marker='o', markersize=4, label="Meta adaptation: mean loss", color=color_meta, alpha=0.8)
    plt.fill_between(
        steps,
        full_min_losses_meta,
        full_max_losses_meta,
        alpha=0.15,
        edgecolor="none",
        color=color_meta,
        label="Meta adaptation: min/max range"
    )

    color_joint = "#2E7D32" 
    plt.plot(steps, full_mean_losses_joint, marker='o', markersize=4, label="Joint adaptation: mean loss", color=color_joint, alpha=0.8)
    plt.fill_between(
        steps,
        full_min_losses_joint,
        full_max_losses_joint,
        alpha=0.15,
        edgecolor="none",
        color=color_joint,
        label="Joint adaptation: min/max range"
    )

    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Full Test Loss")
    plt.xlim(0, 5)
    plt.title("Performance on Entire Test Distribution")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

In [151]:
# ADAPTATION ON TEST TASK WITH META-LEARNED POLICY
adapted_policy_from_meta_learning, adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta = maml_adapt(
    meta_policy,
    A_d, B_tilde_d,
    x0_test, disturbances_test,
    lr_inner=5e-2,
    nb_gradient_descent=10,
    K_samples=5,
    device="cpu"
)

/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:452: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(theta_policy, theta_adapt, (x,))


[Adapt 1] inner_loss=25918084.0000, full_mean=5331222.5000
[Adapt 2] inner_loss=4561670.5000, full_mean=417285.0312
[Adapt 3] inner_loss=374556.3125, full_mean=749.7514
[Adapt 4] inner_loss=731.8719, full_mean=755.7472
[Adapt 5] inner_loss=775.4564, full_mean=627.1074
[Adapt 6] inner_loss=616.7922, full_mean=714.7651
[Adapt 7] inner_loss=732.2876, full_mean=601.7894
[Adapt 8] inner_loss=592.8089, full_mean=688.1091
[Adapt 9] inner_loss=703.1631, full_mean=581.1761
[Adapt 10] inner_loss=578.3618, full_mean=641.6322


In [198]:
plot_adapted_losses(adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, filename="results_all_directions/meta_adaptation.png")

In [ ]:
# APAPTATION ON TEST TASK WITH JOINTLY TRAINED POLICY
adapted_policy_from_joint_learning, adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint = maml_adapt(
    joint_policy,
    A_d, B_tilde_d,
    x0_test, disturbances_test,
    lr_inner=1e-2,
    nb_gradient_descent=10,
    K_samples=5,
    device="cpu"
)

/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:452: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(theta_policy, theta_adapt, (x,))


[Adapt 1] inner_loss=134448.5781, full_mean=68660.3125
[Adapt 2] inner_loss=69368.8672, full_mean=27103.8301
[Adapt 3] inner_loss=25895.6289, full_mean=4805.7139
[Adapt 4] inner_loss=4630.1025, full_mean=4233.1558
[Adapt 5] inner_loss=3581.4622, full_mean=2311.6902
[Adapt 6] inner_loss=2177.8472, full_mean=8100.2007
[Adapt 7] inner_loss=2625.4033, full_mean=27965.7402
[Adapt 8] inner_loss=83121.8906, full_mean=8438.9395
[Adapt 9] inner_loss=4544.5181, full_mean=2144.0801
[Adapt 10] inner_loss=1835.1119, full_mean=10668.5654


In [199]:
plot_adapted_losses(adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, filename="results_all_directions/joint_adaptation.png")

In [317]:
# COMPARE ADAPTED POLICIES ON TEST TASK
plot_adaptation_comparison(full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, "results_all_directions/adaptation_comparison.png")

In [200]:
# TESTING ADAPTED POLICY FROM META-LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(meta_policy, adapted_policy_from_meta_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_meta, U_test_adapted_from_meta = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device)
)
costs = traj_cost(X_test_adapted_from_meta, U_test_adapted_from_meta)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_meta, filename="results_all_directions/testing_xyz_adapted_from_meta.png")
plot_angles_mean(X_test_adapted_from_meta, filename="results_all_directions/testing_angles_adapted_from_meta.png")
plot_controls_grid_mean(U_test_adapted_from_meta, filename="results_all_directions/testing_controls_adapted_from_meta.png")
animate_trajectories(X_test_adapted_from_meta, "results_all_directions/testing_xyz_3d_adapted_from_meta.mp4")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_26100/196220121.py:5: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(meta_policy, adapted_policy_from_meta_learning, (x,))


Adapted test cost = 641.6322021484375
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [201]:
# TESTING ADAPTED POLICY FROM JOINT LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(joint_policy, adapted_policy_from_joint_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_joint, U_test_adapted_from_joint = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device)
)
costs = traj_cost(X_test_adapted_from_joint, U_test_adapted_from_joint)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_joint, filename="results_all_directions/testing_xyz_adapted_from_joint.png")
plot_angles_mean(X_test_adapted_from_joint, filename="results_all_directions/testing_angles_adapted_from_joint.png")
plot_controls_grid_mean(U_test_adapted_from_joint, filename="results_all_directions/testing_controls_adapted_from_joint.png")
animate_trajectories(X_test_adapted_from_joint, "results_all_directions/testing_xyz_3d_adapted_from_joint.mp4")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_26100/2982914859.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_26100/2982914859.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_26100/2982914859.py:5: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  re

Adapted test cost = 10668.5654296875
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [318]:
# Compute improvement percentage plot
meta = np.array(full_mean_losses_meta)
joint = np.array(full_mean_losses_joint)

# Percentage improvement of meta vs joint
improvement_pct = 100 * (joint - meta) / joint

steps = np.arange(0, len(improvement_pct))

plt.figure(figsize=(6,4))
plt.plot(steps, improvement_pct, marker='o', markersize=4, color='purple', alpha=0.8)

plt.xlabel("Adaptation Step")
plt.ylabel("Loss improvement (\%)")
plt.ylim(0, 100)
plt.xlim(0, 5)
plt.title("Improvement of Meta-Learning vs Joint Training")
plt.grid(True)
plt.tight_layout()
plt.savefig("results_all_directions/improvement_percentage.png", dpi=300)
plt.close()

In [ ]:
# Robustness: adaptation with different 5 samples from test distribution

all_robust_full_mean_losses_joint = []
all_robust_full_mean_losses_meta = []

for i in range(10):
    print(f"\n=== ROBUSTNESS TEST {i+1} ===")

    # APAPTATION ON TEST TASK WITH JOINTLY TRAINED POLICY
    robust_adapted_policy_from_joint_learning, robust_adapt_losses_joint, robust_full_mean_losses_joint, robust_full_min_losses_joint, robust_full_max_losses_joint = maml_adapt(
        joint_policy,
        A_d, B_tilde_d,
        x0_test, disturbances_test,
        lr_inner=1e-2,
        nb_gradient_descent=6,
        K_samples=5,
        device="cpu"
    )
    all_robust_full_mean_losses_joint.append(robust_full_mean_losses_joint)


    # ADAPTATION ON TEST TASK WITH META-LEARNED POLICY
    robust_adapted_policy_from_meta_learning, robust_adapt_losses_meta, robust_full_mean_losses_meta, robust_full_min_losses_meta, robust_full_max_losses_meta = maml_adapt(
        meta_policy,
        A_d, B_tilde_d,
        x0_test, disturbances_test,
        lr_inner=5e-2,
        nb_gradient_descent=6,
        K_samples=5,
        device="cpu"
    )
    all_robust_full_mean_losses_meta.append(robust_full_mean_losses_meta)


=== ROBUSTNESS TEST 1 ===
[Adapt 1] inner_loss=134602.2031, full_mean=87814.2578
[Adapt 2] inner_loss=89265.0469, full_mean=32593.4004
[Adapt 3] inner_loss=36925.0234, full_mean=5972.5137
[Adapt 4] inner_loss=6994.7061, full_mean=3361.3118
[Adapt 5] inner_loss=3314.8450, full_mean=2210.2537
[Adapt 6] inner_loss=2456.0918, full_mean=4202.4624
[Adapt 1] inner_loss=25148172.0000, full_mean=2522590.7500
[Adapt 2] inner_loss=2950936.5000, full_mean=372929.1562
[Adapt 3] inner_loss=508332.4062, full_mean=724.6127
[Adapt 4] inner_loss=757.3898, full_mean=799.8309
[Adapt 5] inner_loss=787.2052, full_mean=588.7428
[Adapt 6] inner_loss=621.8012, full_mean=710.0734

=== ROBUSTNESS TEST 2 ===
[Adapt 1] inner_loss=127517.2266, full_mean=70538.3438
[Adapt 2] inner_loss=63068.1367, full_mean=32689.6348
[Adapt 3] inner_loss=28170.4941, full_mean=8905.2998
[Adapt 4] inner_loss=7836.0767, full_mean=3426.0569
[Adapt 5] inner_loss=3153.6558, full_mean=1529.3844
[Adapt 6] inner_loss=1503.1251, full_mean=7

In [ ]:
# Plot all robustness curves
plt.figure(figsize=(8, 5))

color_joint = "#2E7D32"   # forest green
color_meta  = "#D81B60"   # magenta

# JOINT TRAINING CURVES (5 curves)
for i, losses in enumerate(all_robust_full_mean_losses_joint):
    plt.plot(
        losses,
        color=color_joint,
        alpha=0.6,
        linewidth=2,
        label="Joint adaptation with 5 samples from test distribution" if i == 0 else None
    )

# META-LEARNING CURVES (5 curves)
for i, losses in enumerate(all_robust_full_mean_losses_meta):
    plt.plot(
        losses,
        color=color_meta,
        alpha=0.6,
        linewidth=2,
        label="Meta adaptation with 5 samples from test distribution" if i == 0 else None
    )

plt.xlabel("Adaptation step")
plt.ylabel("Mean full test loss")
plt.yscale("log")
plt.xlim(0, 5)
plt.grid(True, ls="--", lw=0.4)
plt.legend()
plt.tight_layout()
plt.savefig("results_all_directions/robustness_adaptation_comparison.png", dpi=300)
plt.close()